In [1]:
import pandas as pd
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty

df = pd.read_parquet("../data/li_electrodes_clean.parquet")
print(df.shape)

# Step 1: string to pymatgen Composition object
stc = StrToComposition()
df = stc.featurize_dataframe(df, "framework_formula")
# This adds a "composition" column of Composition objects
df[["framework_formula", "composition"]].head()

(2634, 22)


StrToComposition:   0%|          | 0/2634 [00:00<?, ?it/s]

,framework_formula,composition
0,Sb,(Sb)
1,C,(C)
2,Bi,(Bi)
3,Ag,(Ag)
4,C,(C)


In [2]:
print("rows:", len(df))
print("nulls in composition:", df["composition"].isna().sum())
# how many frameworks are single-element?
n_single = (df["composition"].apply(len) == 1).sum()
print(f"single-element frameworks: {n_single} ({n_single/len(df)*100:.1f}%)")

rows: 2634
nulls in composition: 0
single-element frameworks: 8 (0.3%)


In [3]:
ep = ElementProperty.from_preset("magpie")
df = ep.featurize_dataframe(df, "composition")
print(df.shape)

ElementProperty:   0%|          | 0/2634 [00:00<?, ?it/s]

(2634, 155)


In [4]:
df.filter(like="MagpieData").isna().sum().sum()

np.int64(0)

In [5]:
# Feature matrix: only the Magpie columns
feature_cols = [c for c in df.columns if c.startswith("MagpieData")]
X = df[feature_cols]
y = df["average_voltage"]

print("X shape:", X.shape)   # (2634, 132)
print("y shape:", y.shape)

X shape: (2634, 132)
y shape: (2634,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")

Train: 2107  Test: 527


In [7]:
import numpy as np
from sklearn.metrics import mean_absolute_error

# The dumbest possible model: always predict the training mean
baseline_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Dummy baseline MAE (predict mean): {baseline_mae:.3f} V")

Dummy baseline MAE (predict mean): 0.880 V


In [8]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

model = LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"LightGBM  MAE:  {mae:.3f} V")
print(f"LightGBM  RMSE: {rmse:.3f} V")
print(f"LightGBM  R²:   {r2:.3f}")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005359 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9893
[LightGBM] [Info] Number of data points in the train set: 2107, number of used features: 118
[LightGBM] [Info] Start training from score 3.434716
LightGBM  MAE:  0.486 V
LightGBM  RMSE: 0.697 V
LightGBM  R²:   0.601


In [9]:
import pandas as pd

# Clean the whitespace in feature names for readable plots
X_train.columns = X_train.columns.str.replace(" ", "_")
X_test.columns = X_test.columns.str.replace(" ", "_")
# Refit on cleaned names so importances map correctly
model.fit(X_train, y_train)

importances = pd.Series(model.feature_importances_, index=X_train.columns)
top15 = importances.sort_values(ascending=False).head(15)
print(top15)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022843 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9893
[LightGBM] [Info] Number of data points in the train set: 2107, number of used features: 118
[LightGBM] [Info] Start training from score 3.434716
MagpieData_avg_dev_SpaceGroupNumber     663
MagpieData_avg_dev_Number               533
MagpieData_avg_dev_NValence             517
MagpieData_mean_NUnfilled               512
MagpieData_mean_GSvolume_pa             511
MagpieData_avg_dev_Electronegativity    508
MagpieData_avg_dev_CovalentRadius       471
MagpieData_mean_Electronegativity       470
MagpieData_mean_SpaceGroupNumber        465
MagpieData_mean_NValence                444
MagpieData_avg_dev_NUnfilled            442
MagpieData_avg_dev_MeltingT             431
MagpieData_mean_CovalentRadius          426
MagpieData_mean_MeltingT                416
MagpieData_mean_GSmagmom                41

In [10]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, X_test, y_test,
    n_repeats=10, random_state=42, scoring="neg_mean_absolute_error",
)
perm = pd.Series(result.importances_mean, index=X_test.columns)
print(perm.sort_values(ascending=False).head(15))

MagpieData_mean_CovalentRadius         0.152174
MagpieData_mean_Column                 0.073531
MagpieData_mean_GSvolume_pa            0.030374
MagpieData_avg_dev_Number              0.023466
MagpieData_mean_Electronegativity      0.021209
MagpieData_range_CovalentRadius        0.018969
MagpieData_avg_dev_SpaceGroupNumber    0.017294
MagpieData_avg_dev_NpUnfilled          0.015500
MagpieData_maximum_GSmagmom            0.011104
MagpieData_mean_SpaceGroupNumber       0.010675
MagpieData_maximum_CovalentRadius      0.010082
MagpieData_mean_Row                    0.009590
MagpieData_mean_GSmagmom               0.007835
MagpieData_range_Number                0.007475
MagpieData_avg_dev_NUnfilled           0.006679
dtype: float64


In [11]:
from sklearn.model_selection import cross_val_score

cv_mae = -cross_val_score(
    model, X, y, cv=5, scoring="neg_mean_absolute_error"
)
print(f"5-fold CV MAE: {cv_mae.mean():.3f} ± {cv_mae.std():.3f} V")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004716 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9528
[LightGBM] [Info] Number of data points in the train set: 2107, number of used features: 116
[LightGBM] [Info] Start training from score 3.506650
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002315 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9971
[LightGBM] [Info] Number of data points in the train set: 2107, number of used features: 119
[LightGBM] [Info] Start training from score 3.295555
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of te